# 09 · Modelo final — única evaluación en Test

**Qué responde.** Cierra la subsanación: evalúa **una sola vez**, en un conjunto de prueba nunca usado para decidir nada, el modelo elegido en validación, y lo compara con la persistencia (pedido 5.3 del especialista), la climatología y el modelo desplegado en la aplicación (v2.0).

> ⚠️ **Regla del protocolo:** el conjunto de prueba se evalúa una sola vez. Si se reentrena o se cambia algo después de ver estos resultados, el Test deja de ser independiente. Este notebook solo **lee** los resultados del script.

| Elemento | Valor |
|---|---|
| Modelo | LSTM apilada 64/64 · dropout 0.3 · Dense 64 → 20 |
| Entradas | S4 + hora del día + día del año (seno y coseno) |
| Ventana / horizonte | 60 min / 20 min |
| Pérdida | Weighted Focal Loss · α = 1.5 · β = 10 |
| Umbral de alerta | El calibrado en validación por la semilla elegida (≈ 0.55) |
| Test | 2 mar 2026 – 21 jul 2026 (125 días, 13 activos) |
| Comparación justa | Todos los modelos se evalúan sobre **los mismos instantes objetivo** (ventanas válidas a la vez para 60 y 360 min) |

## 1. De dónde sale cada decisión

| Decisión | Valor | Evidencia |
|---|---|---|
| Partición | Cronológica Out-of-Time, corte a las 07 LT | Notebook 02 |
| Arquitectura | LSTM apilada | Torneo repetido 3 veces (tesis, sección de arquitecturas) |
| Ventana | 60 min | Notebook 03 (sensibilidad) y notebook 08 (confirmación) |
| Función de pérdida | WFL (decisiva: HSS 0.15 → 0.56) | Notebook 05 (ablación) |
| α, β | 1.5, 10 | Notebook 06 (tres fases, umbral calibrado) |
| LSTM frente a Transformer | LSTM (HSS 0.66 frente a 0.60) | Notebook 07 |
| Variables | S4 + hora, sin ROTI | Notebook 08 (explicabilidad y confirmación) |

## 2. Script empleado: `scripts/correr_modelo_final.py`

```bash
nohup python scripts/correr_modelo_final.py > experiments/log_modelo_final.txt 2>&1 &
```

Pasos que ejecuta:

1. Reentrena la configuración final con 3 semillas; elige la semilla con mayor HSS calibrado **en validación** y fija su umbral.
2. Evalúa **una sola vez en Test**: modelo final, persistencia, climatología diaria y modelo desplegado v2.0 (`app/produccion_actual`), sobre los mismos instantes objetivo.
3. Significancia frente a cada referencia: bootstrap por días (HSS y RMSE en eventos; para el HSS, cada modelo con su propio umbral) y Diebold–Mariano.
4. Análisis por fase del evento y de falsas alarmas; latencia en CPU.
5. Exporta `modelo_S4_lstm.keras`, `SCALER_S4_app.pkl` y `config.json` a `app/modelo_final/`, listos para la aplicación.

## 3. Reporte (no entrena)

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, ruta
from s4.registry import leer, resumir_semillas
cfg = cargar_config("../config/ablacion_2025_2026.yaml")
REG = ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv"
FIG, TAB = ruta(cfg, "reportes") / "figures", ruta(cfg, "reportes") / "tables"
AZUL, CELESTE, GRIS, AMBAR, ROJO = "#0000AF", "#0099FF", "#B8BED6", "#C77C02", "#C62828"

In [ ]:
final = pd.read_csv(TAB / "tabla_final_test.csv")
cols = ["modelo", "umbral_alerta", "RMSE_global", "RMSE_evento", "POD", "Precision", "FAR", "HSS", "POD_inicio", "antelacion_media_min"]
display(final[cols].round(3))
sig = pd.read_csv(TAB / "tabla_final_significancia.csv"); display(sig.round(4))

In [ ]:
orden = [m for m in ["Persistencia", "Climatología diaria", "Modelo desplegado v2.0", "Modelo final"] if m in set(final["modelo"])]
col = {"Persistencia": AMBAR, "Climatología diaria": GRIS, "Modelo desplegado v2.0": CELESTE, "Modelo final": AZUL}
f = final.set_index("modelo").reindex(orden)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
for ax, (m, tit) in zip(axes, [("HSS", "Habilidad de alerta (HSS)"), ("POD_inicio", "Inicios anticipados"), ("Precision", "Precisión de la alerta"), ("RMSE_evento", "RMSE en eventos (menor es mejor)")]):
    v = f[m].fillna(0)
    ax.bar(range(len(orden)), v, color=[col[o] for o in orden])
    for i, x in enumerate(v): ax.text(i, x, f"{x:.2f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(range(len(orden)), [o.replace(" ", "\n", 1) for o in orden], fontsize=9)
    ax.set_title(tit, fontsize=11); ax.set_ylim(0, max(v.max() * 1.2, 0.05)); ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("Evaluación final en Test (2 mar – 21 jul 2026), mismos instantes objetivo para todos los modelos")
plt.tight_layout(); plt.savefig(FIG / "fig_final_test.png", dpi=200); plt.show()

In [ ]:
fases = pd.read_csv(TAB / "tabla_final_fases.csv"); display(fases.pivot_table(index="fase", columns="modelo", values="tasa_alerta").round(3))
display(pd.read_csv(TAB / "tabla_final_falsas_alarmas.csv"))

### 3.1 Caso de estudio: el evento más intenso del periodo de prueba

In [ ]:
z = np.load(TAB / "predicciones_test.npz")
t = pd.to_datetime(z["t"].astype("datetime64[m]"))
y20 = pd.Series(z["y"][:, -1], index=t + pd.Timedelta(minutes=20))
dia = y20.idxmax().floor("D")
ini, fin = dia - pd.Timedelta(hours=2), dia + pd.Timedelta(hours=10)
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(y20.loc[ini:fin], color="black", lw=1, label="S4 observado")
for clave, etiqueta, c in [("Modelo_final", "Modelo final (+20 min)", AZUL), ("Persistencia", "Persistencia (+20 min)", AMBAR), ("Modelo_desplegado_v20", "Modelo desplegado v2.0 (+20 min)", CELESTE)]:
    if clave in z.files:
        ax.plot(pd.Series(z[clave][:, -1], index=y20.index).loc[ini:fin], color=c, lw=1.6, label=etiqueta, alpha=0.9)
ax.axhline(0.6, color=ROJO, ls="--", lw=1.2, label="Umbral S4 = 0.6")
ax.set_ylabel("S4"); ax.set_xlabel("Hora (UT)"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title(f"Caso de estudio en Test: {dia:%d-%m-%Y} (pronósticos dibujados en el instante pronosticado)")
plt.tight_layout(); plt.savefig(FIG / "fig_final_caso_estudio.png", dpi=200); plt.show()

## 4. Resultados obtenidos e interpretación

**Test:** 2 mar – 21 jul 2026 · 162 583 ventanas comunes · 2 251 con evento · 1 804 inicios de evento. Semilla elegida en validación: 7 (HSS calibrado 0.667, umbral 0.55, igual en las 3 semillas).

| Modelo | Umbral | HSS | POD | Precisión | Inicios anticipados | Antelación | RMSE eventos | RMSE global |
|---|---|---|---|---|---|---|---|---|
| **Modelo final** | 0.55 | **0.545** | **44 %** | 74 % | **33 %** | 5.1 min | **0.223** | **0.048** |
| Modelo desplegado v2.0 | 0.65 | 0.451 | 32 % | 77 % | 21 % | 5.1 min | 0.249 | 0.051 |
| Persistencia | 0.60 | 0.321 | 20 % | 92 % | 0 % | — | 0.304 | 0.051 |
| Climatología diaria | 0.60 | 0.000 | 0 % | — | 0 % | — | 0.596 | 0.061 |

**Significancia del modelo final** (bootstrap por días, IC 95 %; Diebold–Mariano sobre el MSE global; positivo = mejora del modelo final):

| Frente a | Mejora de HSS | Mejora de RMSE en eventos | Diebold–Mariano |
|---|---|---|---|
| Persistencia | **+0.224** [0.127; 0.291] | **+0.082** [0.042; 0.098] | DM = 1.73, p = 0.086 |
| Modelo desplegado v2.0 | **+0.094** [0.009; 0.167] | **+0.026** [0.012; 0.031] | **DM = 2.06, p = 0.042** |

**Por fase:** en los 1 804 inicios de evento, el modelo final alertó el 32.5 % (v2.0: 21.5 %; persistencia: 0 %). En calma alerta en el 0.2 % de las ventanas.

**Falsas alarmas:** la tasa baja a 26 % (frente a 55 % del modelo v2.0 en su propia evaluación); ninguna con S4 < 0.2 y el 81 % con S4 real entre 0.4 y 0.6 (51 % entre 0.5 y 0.6).

### Conclusiones

1. **El modelo final supera significativamente a la persistencia** en habilidad de alerta (+0.22 de HSS) y en error en eventos (−27 %), y anticipa un tercio de los inicios de evento, que la persistencia no puede anticipar.
2. **Desaparece el costo en calma:** su RMSE global (0.048) es incluso menor que el de la persistencia (0.051); Diebold–Mariano favorece al modelo final, aunque sin significancia al 5 % (p = 0.086). Con β = 100, el modelo anterior era significativamente peor que la persistencia en error global; con β = 10 ese costo se elimina.
3. **Supera significativamente al modelo desplegado v2.0** en las mismas ventanas: +0.09 de HSS, más inicios anticipados (33 % frente a 21 %) y menor error, con un modelo más simple: 60 min de historia en vez de 360 y sin precursores externos.
4. **Las falsas alarmas son pocas y físicamente razonables:** una de cada cuatro alertas, casi siempre con centelleo moderado cercano al umbral.
5. **Limitación:** el Test (mar–jul 2026) incluye el invierno con poca actividad (13 días activos); la anticipación de inicios en Test (33 %) es menor que en validación (59 %), lo que indica que el desempeño depende de la temporada y debe seguir monitoreándose en operación.

## 5. Despliegue

Los archivos de `app/modelo_final/` (`modelo_S4_lstm.keras`, `SCALER_S4_app.pkl`, `config.json`) reemplazan a los de la aplicación; el `app.py` es el mismo. La aplicación leerá de `config.json` la nueva ventana (60 min), el umbral calibrado (0.55) y las métricas de Test. El modelo anterior se conserva en `app/produccion_actual/` como versión 2.0.